In [ ]:
# ── Setup ────────────────────────────────────────────────────────
import os
import warnings
import time
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── Import multi-agent wiring ────────────────────────────────────
from knowlytix.harness.governance.agents.multi_agent import (
    AgentRole, GovernedAgent, MultiAgentOrchestrator,
    DATA_AGENT_ROLE, TRAINING_AGENT_ROLE,
    DIAGNOSTICS_AGENT_ROLE, FAIRNESS_AGENT_ROLE,
)
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.memory import AgentMemory

print("=== Agent Roles ===")
for role in [DATA_AGENT_ROLE, TRAINING_AGENT_ROLE,
             DIAGNOSTICS_AGENT_ROLE, FAIRNESS_AGENT_ROLE]:
    print(f"\n  {role.name}:")
    print(f"    Allowed:   {sorted(role.allowed_tools)}")
    print(f"    Forbidden: {sorted(role.forbidden_tools)}")
    print(f"    {role.description}")

## Key Concepts

| Term | Definition |
|------|------------|
| **Multi-agent governance** | Decomposing authority into roles, enforcing those roles structurally and auditing handoffs between roles. |
| **AgentRole** | A named role with an *allowed-tool* set, a *forbidden-tool* set and a human-readable description. Permission checks are set-membership tests, not prompt-based hints. |
| **GovernedAgent** | Wraps a role, a shared `SessionState` and a private `AgentMemory`. Every `execute(tool, kwargs)` call checks the role's allowed set before dispatching. |
| **HandoffRecord** | Captures the from-agent, to-agent and the datasets, models and results transferred at each phase transition. |
| **Domain-store isolation** | Each agent writes triples into its own GMS store. Cross-agent knowledge transfer occurs only through the shared `SessionState` (for artifacts) or explicit cross-store queries (for knowledge). |
| **ConsensusOracle** | Queries all domain stores for a given entity, reporting coverage (fraction of stores that know it) and per-agent triple counts. |
| **Separation of duty** | No single actor controls an entire end-to-end process; each pipeline phase is assigned to a dedicated agent with minimal permissions. |
| **ENM (Entity-Numeric Map)** | Key-value store that attaches scalar metrics to entities within a GMS store (see Ch 5). |

## Lab 17.1 -- Run a 4-Agent Governed Pipeline

Each agent gets its own `AgentMemory` (separate domain GMS store) but all
four share a single `SessionState` so data and models flow between phases.

**Pipeline:** DataAgent -> TrainingAgent -> DiagnosticsAgent -> FairnessAgent

In [3]:
# ── Step 1: Create 4 agents with SEPARATE domain stores ─────────
import shutil

STORE_BASE = os.path.join(os.getcwd(), "governance_monograph", "lab_stores", "ch17_multi_agent")

# Clean previous runs for reproducibility
for sub in ["data", "training", "diagnostics", "fairness"]:
    p = os.path.join(STORE_BASE, sub)
    if os.path.exists(p):
        shutil.rmtree(p)

# Shared session -- all agents see the same datasets and models
shared_session = SessionState()

# Each agent gets its own AgentMemory (separate domain GMS)
data_memory = AgentMemory(
    os.path.join(STORE_BASE, "data"), device=str(DEVICE))
training_memory = AgentMemory(
    os.path.join(STORE_BASE, "training"), device=str(DEVICE))
diagnostics_memory = AgentMemory(
    os.path.join(STORE_BASE, "diagnostics"), device=str(DEVICE))
fairness_memory = AgentMemory(
    os.path.join(STORE_BASE, "fairness"), device=str(DEVICE))

# Build GovernedAgent instances
data_agent = GovernedAgent(DATA_AGENT_ROLE, shared_session, data_memory)
training_agent = GovernedAgent(TRAINING_AGENT_ROLE, shared_session, training_memory)
diagnostics_agent = GovernedAgent(DIAGNOSTICS_AGENT_ROLE, shared_session, diagnostics_memory)
fairness_agent = GovernedAgent(FAIRNESS_AGENT_ROLE, shared_session, fairness_memory)

print("=== 4 GovernedAgents Created ===")
for ag in [data_agent, training_agent, diagnostics_agent, fairness_agent]:
    print(f"  {ag.role.name:20s}  store={ag.memory._config.store_path}")
print(f"\nShared session id: {id(shared_session)}")
print(f"All agents share same session: "
      f"{data_agent.session is training_agent.session is diagnostics_agent.session is fairness_agent.session}")

  GMS entities:  2
  GMS relations: 1
  GMS triples:   1



Training GMS: 2 entities, 1 relations, 1 triples
  Config: d_v=128, m=64, epochs=5, batch=256, neg=32
  Device: cuda


  Epoch    1/5  Loss: 8.1211  (L_pos=1.3741, L_neg=0.5003)
  GMS entities:  2
  GMS relations: 1
  GMS triples:   1

Training GMS: 2 entities, 1 relations, 1 triples
  Config: d_v=128, m=64, epochs=5, batch=256, neg=32
  Device: cuda
  Epoch    1/5  Loss: 8.9122  (L_pos=1.5881, L_neg=0.3888)
  GMS entities:  2
  GMS relations: 1
  GMS triples:   1

Training GMS: 2 entities, 1 relations, 1 triples
  Config: d_v=128, m=64, epochs=5, batch=256, neg=32
  Device: cuda
  Epoch    1/5  Loss: 9.5546  (L_pos=1.7079, L_neg=0.4060)
  GMS entities:  2
  GMS relations: 1
  GMS triples:   1

Training GMS: 2 entities, 1 relations, 1 triples
  Config: d_v=128, m=64, epochs=5, batch=256, neg=32
  Device: cuda
  Epoch    1/5  Loss: 8.8156  (L_pos=1.4942, L_neg=0.5378)
=== 4 GovernedAgents Created ===
  DataAgent             store=/home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/data
  TrainingAgent         store=/home/asudjianto/jupyterlab/kg-memory/governance_monog

In [4]:
# ── Step 2: Wire the orchestrator ────────────────────────────────
agents = {
    "DataAgent": data_agent,
    "TrainingAgent": training_agent,
    "DiagnosticsAgent": diagnostics_agent,
    "FairnessAgent": fairness_agent,
}

orchestrator = MultiAgentOrchestrator(agents, shared_session)
print(f"Orchestrator: {len(agents)} agents registered")
for name in agents:
    print(f"  {name}")

Orchestrator: 4 agents registered
  DataAgent
  TrainingAgent
  DiagnosticsAgent
  FairnessAgent


In [ ]:
# ── Step 3: Define the pipeline phases ───────────────────────────
# Phase 1: DataAgent loads, summarizes, correlates and preprocesses Adult Income
# Phase 2: TrainingAgent trains 5 classifiers and builds the leaderboard
# Phase 3: DiagnosticsAgent runs robustness + feature importance for top 2
# Phase 4: FairnessAgent runs fairness on sex and race for the top model
#
# We reuse the canonical Adult Income phase helpers from
# governed_agent.agents.data_agent so the pipeline harmonizes with
# Ch 6/7/8/10/12/15/16. Each helper returns a list[(tool_name, args)]
# without touching the orchestrator --- it is just a phase definition.
from knowlytix.harness.governance.agents.data_agent import (
    adult_data_phase, adult_training_phase,
    adult_diagnostics_phase, adult_fairness_phase,
)

pipeline_phases = [
    ("DataAgent", adult_data_phase()),
    ("TrainingAgent", adult_training_phase()),
    ("DiagnosticsAgent", adult_diagnostics_phase(top_models=["GB", "RF"])),
    ("FairnessAgent", adult_fairness_phase(top_model="GB")),
]

print(f"Pipeline: {len(pipeline_phases)} phases over Adult Income")
for agent_name, calls in pipeline_phases:
    print(f"  {agent_name:18s}: {[c[0] for c in calls]}")

In [6]:
# ── Step 4: Execute the pipeline ─────────────────────────────────
t0 = time.time()
result = orchestrator.run_pipeline(pipeline_phases)
elapsed = time.time() - t0

print(f"=== Pipeline Complete ({elapsed:.1f}s) ===")
print(f"  Total tool calls: {result.total_tools}")
print(f"  Total triples:    {result.total_triples}")
print(f"  Total handoffs:   {result.total_handoffs}")

  [BackgroundTrain] Starting: 10 entities, 6 relations, 9 triples


  [BackgroundTrain] Done: Trained: 10 entities, 6 relations, 50 steps in 5.1s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/data
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 5 entities, 9 relations, 9 triples


  [BackgroundTrain] Done: Trained: 5 entities, 9 relations, 50 steps in 5.0s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/training
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 9 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.8s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/training
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 1 entities, 1 relations, 1 triples


  [BackgroundTrain] Done: Trained: 1 entities, 1 relations, 50 steps in 3.5s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/training
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 3 entities, 2 relations, 19 triples


  [BackgroundTrain] Done: Trained: 3 entities, 2 relations, 50 steps in 4.1s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/training
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 5 entities, 6 relations, 6 triples


  [BackgroundTrain] Done: Trained: 5 entities, 6 relations, 50 steps in 5.0s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/diagnostics
  [BackgroundTrain] Store saved


  [BackgroundTrain] Starting: 4 entities, 5 relations, 6 triples


  [BackgroundTrain] Done: Trained: 4 entities, 5 relations, 50 steps in 4.3s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/diagnostics
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 9 entities, 26 relations, 26 triples


  [BackgroundTrain] Done: Trained: 9 entities, 26 relations, 50 steps in 5.9s
  Store saved to /home/asudjianto/jupyterlab/kg-memory/governance_monograph/lab_stores/ch17_multi_agent/fairness
  [BackgroundTrain] Store saved
=== Pipeline Complete (37.5s) ===
  Total tool calls: 10
  Total triples:    86
  Total handoffs:   3


In [7]:
# ── Step 5: Show handoff records ─────────────────────────────────
print(orchestrator.format_handoffs())

Handoff Records
  [1] DataAgent -> TrainingAgent
      datasets:['iris']
  [2] TrainingAgent -> DiagnosticsAgent
      datasets:['iris']
      models:['RF_iris', 'GBT_iris']
      results:['RF_iris', 'GBT_iris']
  [3] DiagnosticsAgent -> FairnessAgent
      datasets:['iris']
      models:['RF_iris', 'GBT_iris']
      results:['RF_iris', 'GBT_iris', 'RF_iris_robustness', 'RF_iris_importance']


In [8]:
# ── Step 6: Per-agent statistics ─────────────────────────────────
print(orchestrator.format_agent_stats())

Per-Agent Statistics
  DataAgent:
    Tools called:    2
    Tool names:      ['load_dataset', 'data_summary']
    Triples learned: 9
    Blocked:         0
    Domain store:    entities=12, triples=10, enm=4
  TrainingAgent:
    Tools called:    5
    Tool names:      ['train_model', 'train_model', 'predict', 'predict', 'leaderboard']
    Triples learned: 39
    Blocked:         0
    Domain store:    entities=13, triples=40, enm=32
  DiagnosticsAgent:
    Tools called:    2
    Tool names:      ['diagnose_robustness', 'explain_importance']
    Triples learned: 12
    Blocked:         0
    Domain store:    entities=11, triples=13, enm=7
  FairnessAgent:
    Tools called:    1
    Tool names:      ['diagnose_fairness']
    Triples learned: 26
    Blocked:         0
    Domain store:    entities=11, triples=27, enm=23


In [9]:
# ── Step 7: Domain store isolation -- each store is distinct ─────
print("=== Domain Store Sizes ===")
for name, ag in agents.items():
    ds = ag.domain_stats()
    print(f"  {name:20s}  entities={ds.get('entities', 0):3d}  "
          f"triples={ds.get('triples', 0):3d}  "
          f"enm={ds.get('enm_entries', 0):3d}")

# Verify physical isolation
print("\n=== Store Object Identity ===")
stores = {name: id(ag.memory.store) for name, ag in agents.items()}
all_distinct = len(set(stores.values())) == len(stores)
for name, sid in stores.items():
    print(f"  {name:20s}  id={sid}")
print(f"\nAll stores distinct: {all_distinct}")

=== Domain Store Sizes ===
  DataAgent             entities= 12  triples= 10  enm=  4
  TrainingAgent         entities= 13  triples= 40  enm= 32
  DiagnosticsAgent      entities= 11  triples= 13  enm=  7
  FairnessAgent         entities= 11  triples= 27  enm= 23

=== Store Object Identity ===
  DataAgent             id=269807358005696
  TrainingAgent         id=269806681093136
  DiagnosticsAgent      id=269806680334912
  FairnessAgent         id=269806680329344

All stores distinct: True


## Lab 17.2 -- Enforce Inter-Agent Boundaries

We systematically test that role permissions are **structural**: every
forbidden tool call is blocked by a `PermissionError`, not merely warned.

In [ ]:
# ── Boundary Test 1: TrainingAgent cannot call load_dataset ──────
print("=== Boundary Test 1: TrainingAgent -> load_dataset ===")
blocked, msg = orchestrator.test_boundary(
    "TrainingAgent", "load_dataset",
    {"name": "adult", "builtin": "adult"},
)
print(f"  Blocked: {blocked}")
print(f"  Message: {msg}")

# ── Boundary Test 2: DataAgent cannot call train_model ───────────
print("\n=== Boundary Test 2: DataAgent -> train_model ===")
blocked2, msg2 = orchestrator.test_boundary(
    "DataAgent", "train_model",
    {"dataset": "adult", "model_type": "RandomForestClassifier",
     "model_name": "sneaky_rf"},
)
print(f"  Blocked: {blocked2}")
print(f"  Message: {msg2}")

# ── Boundary Test 3: DiagnosticsAgent cannot train ───────────────
print("\n=== Boundary Test 3: DiagnosticsAgent -> train_model ===")
blocked3, msg3 = orchestrator.test_boundary(
    "DiagnosticsAgent", "train_model",
    {"dataset": "adult", "model_type": "RandomForestClassifier",
     "model_name": "sneaky_rf2"},
)
print(f"  Blocked: {blocked3}")
print(f"  Message: {msg3}")

# ── Boundary Test 4: FairnessAgent cannot load data ──────────────
print("\n=== Boundary Test 4: FairnessAgent -> load_dataset ===")
blocked4, msg4 = orchestrator.test_boundary(
    "FairnessAgent", "load_dataset",
    {"name": "adult", "builtin": "adult"},
)
print(f"  Blocked: {blocked4}")
print(f"  Message: {msg4}")

In [11]:
# ── Boundary Test 5: Exhaustive forbidden-tool matrix ────────────
# Test every agent against every tool in its forbidden set.
boundary_tests = []
for name, ag in agents.items():
    for tool in sorted(ag.role.forbidden_tools):
        blocked_i, msg_i = orchestrator.test_boundary(name, tool, {})
        boundary_tests.append({
            "agent": name, "tool": tool,
            "blocked": blocked_i, "msg": msg_i[:60],
        })

n_total = len(boundary_tests)
n_blocked = sum(1 for t in boundary_tests if t["blocked"])

print("=== Exhaustive Boundary Matrix ===")
print(f"{'Agent':<20s} {'Tool':<25s} {'Blocked':<8s}")
print("-" * 55)
for t in boundary_tests:
    status = "YES" if t["blocked"] else "NO"
    print(f"  {t['agent']:<18s} {t['tool']:<25s} {status:<8s}")

print("\n=== Enforcement Summary ===")
print(f"  Total attempts: {n_total}")
print(f"  Blocked:        {n_blocked}")
print(f"  Enforcement:    {n_blocked/n_total*100:.0f}%")

=== Exhaustive Boundary Matrix ===
Agent                Tool                      Blocked 
-------------------------------------------------------
  DataAgent          diagnose_fairness         YES     
  DataAgent          diagnose_robustness       YES     
  DataAgent          explain_importance        YES     
  DataAgent          explain_pdp               YES     
  DataAgent          explain_shap              YES     
  DataAgent          train_model               YES     
  TrainingAgent      diagnose_fairness         YES     
  TrainingAgent      load_dataset              YES     
  TrainingAgent      preprocess_data           YES     
  DiagnosticsAgent   load_dataset              YES     
  DiagnosticsAgent   preprocess_data           YES     
  DiagnosticsAgent   train_model               YES     
  FairnessAgent      load_dataset              YES     
  FairnessAgent      predict                   YES     
  FairnessAgent      preprocess_data           YES     
  FairnessAge

In [ ]:
# ── Boundary Test 6: Cross-store isolation ───────────────────────
# Verify that writing triples into one agent's memory does NOT
# appear in another agent's memory.

print("=== Cross-Store Isolation Tests ===\n")
pairs = [
    ("DataAgent", "TrainingAgent"),
    ("TrainingAgent", "DiagnosticsAgent"),
    ("DiagnosticsAgent", "FairnessAgent"),
    ("DataAgent", "FairnessAgent"),
]
for a_name, b_name in pairs:
    isolated, msg = MultiAgentOrchestrator.test_store_isolation(
        agents[a_name], agents[b_name],
    )
    status = "PASS" if isolated else "FAIL"
    print(f"  [{status}] {msg}")

# Demonstrate that triples in DataAgent's store are NOT in TrainingAgent's
data_recall = data_agent.memory.recall_entity("adult")
train_recall = training_agent.memory.recall_entity("adult")

print(f"\n  DataAgent recalls 'adult':     exists={data_recall['exists']}, "
      f"head_triples={len(data_recall['triples_as_head'])}")
print(f"  TrainingAgent recalls 'adult':  exists={train_recall['exists']}, "
      f"head_triples={len(train_recall['triples_as_head'])}")
print("\n  Domain stores contain DIFFERENT knowledge about the same entity.")

## Exercise 17.1 -- Consensus Oracle

Build a `ConsensusOracle` that queries all domain stores for a given entity
and reports whether they agree (low tension) or disagree (high tension).

In [ ]:
# ── Exercise 17.1: ConsensusOracle ───────────────────────────────
from dataclasses import dataclass

@dataclass
class ConsensusReport:
    entity: str
    stores_with_entity: list[str]
    stores_without: list[str]
    coverage: float  # fraction of stores that know this entity
    triple_counts: dict[str, int]  # per-store triple count for entity

class ConsensusOracle:
    """Cross-store consensus checker."""
    def __init__(self, agents: dict[str, GovernedAgent]):
        self.agents = agents

    def query(self, entity: str) -> ConsensusReport:
        found, missing = [], []
        counts = {}
        for name, ag in self.agents.items():
            recall = ag.memory.recall_entity(entity)
            if recall["exists"]:
                found.append(name)
                counts[name] = (len(recall["triples_as_head"])
                                + len(recall["triples_as_tail"]))
            else:
                missing.append(name)
                counts[name] = 0
        coverage = len(found) / len(self.agents) if self.agents else 0
        return ConsensusReport(entity, found, missing, coverage, counts)


oracle = ConsensusOracle(agents)

# Query for entities the Adult Income pipeline touched
for entity in ["adult", "GB", "RF", "sex", "race"]:
    report = oracle.query(entity)
    print(f"Entity: {entity!r}")
    print(f"  Coverage:  {report.coverage:.0%} "
          f"({len(report.stores_with_entity)}/{len(agents)})")
    print(f"  Present:   {report.stores_with_entity}")
    print(f"  Absent:    {report.stores_without}")
    print(f"  Triples:   {report.triple_counts}")
    print()

## Exercise 17.2 -- Add a Specialist Agent

Add a fifth agent, `ExplainerAgent`, with allowed tools
`{explain_pdp, explain_shap}` and forbidden tools
`{train_model, load_dataset, diagnose_fairness}`. Run it
on the existing pipeline results.

In [ ]:
# ── Exercise 17.2: Specialist ExplainerAgent ─────────────────────
explainer_role = AgentRole(
    name="ExplainerAgent",
    allowed_tools={"explain_pdp", "explain_shap"},
    forbidden_tools={"train_model", "load_dataset", "diagnose_fairness"},
    description="Runs PDP and SHAP explanations only.",
)

explainer_memory = AgentMemory(
    os.path.join(STORE_BASE, "explainer"), device=str(DEVICE))
explainer_agent = GovernedAgent(explainer_role, shared_session, explainer_memory)

# Run PDP on the top model (GB) using Adult Income features
adult_features = shared_session.datasets["adult"]["feature_names"]
display, raw = explainer_agent.execute("explain_pdp", {
    "dataset": "adult", "model_name": "GB",
    "features": [adult_features[0]],
})
print("ExplainerAgent ran explain_pdp:")
print(f"  {display[:200]}...")
print(f"  Triples learned: {explainer_agent.stats.triples_learned}")

# Confirm ExplainerAgent cannot train
try:
    explainer_agent.execute("train_model", {
        "dataset": "adult", "model_type": "RandomForestClassifier",
        "model_name": "bad_model"})
    print("\n  ERROR: should have been blocked!")
except PermissionError as e:
    print(f"\n  Correctly blocked: {e}")

## Exercise 17.3 -- Cross-Agent Query

Query the TrainingAgent's domain store for the entity "RF_iris" and
the DiagnosticsAgent's domain store for the same entity. Compare
what each knows.

In [ ]:
# ── Exercise 17.3: Cross-agent query ─────────────────────────────
print("=== Cross-Agent Query: 'GB' (top Adult Income model) ===\n")

for agent_name in ["TrainingAgent", "DiagnosticsAgent", "FairnessAgent"]:
    ag = agents[agent_name]
    recall = ag.memory.recall_entity("GB")
    print(f"  {agent_name}:")
    print(f"    exists: {recall['exists']}")
    print(f"    triples as head ({len(recall['triples_as_head'])}):")
    for h, r, t in recall["triples_as_head"][:5]:
        print(f"      ({h}, {r}, {t})")
    if recall["enm_values"]:
        print("    ENM values:")
        for k, v in list(recall["enm_values"].items())[:5]:
            print(f"      {k} = {v}")
    print()

print("Each agent's domain store reflects ONLY the tools it ran:")
print("  TrainingAgent knows accuracy and model type (from train_model)")
print("  DiagnosticsAgent knows robustness scores (from diagnose_robustness)")
print("  FairnessAgent knows AIR by sex / race (from diagnose_fairness)")

## Summary

**Lab 17.1** demonstrated a 4-agent pipeline (DataAgent, TrainingAgent,
DiagnosticsAgent, FairnessAgent) running on **Adult Income** with separate
domain GMS stores and a shared SessionState. Handoff records provide
an audit trail.

**Lab 17.2** confirmed 100% enforcement of role boundaries: every
forbidden tool call was blocked by a structural `PermissionError`, and
cross-store isolation was verified by object identity checks.

**Exercises** introduced the ConsensusOracle (cross-store consensus),
a specialist ExplainerAgent and cross-agent entity queries.

# Chapter 17 -- Multi-Agent Data Science Team

This notebook demonstrates a **4-agent governed pipeline** where each agent
has its own domain GMS (capability boundary) but shares a single
SessionState (accountability).

**Agents:**
- **DataAgent** -- loads and prepares data
- **TrainingAgent** -- trains models and builds leaderboards
- **DiagnosticsAgent** -- runs robustness and explainability diagnostics
- **FairnessAgent** -- runs fairness checks

**Key property:** Role permissions are *structural* (set-based check), not advisory.
A tool call outside the allowed set raises `PermissionError`.

---
## Lab 17.3 --- Calibrated Baselines and Three-State Co-Attestor

Role permissions are declared, not numeric thresholds. What gets
calibrated is the per-agent tool-call frequency baseline (PSI
reference) and the per-role fingerprints (SHA-256 over the
canonical role payload). The bundle ships both, plus the
exhaustive invariants proof.

In [ ]:
os.environ.setdefault("GOVERNED_AGENT_SIGNING_KEY",
                      "lab17-notebook-key-replace-in-deployment")

from knowlytix.harness.governance import (
    fingerprint_orchestrator, MultiAgentCoAttestor, PerAgentToolFrequency,
)

# Per-role fingerprints
fps = fingerprint_orchestrator(orchestrator)
print("Per-role fingerprints (SHA-256 over canonical payload):")
for role, fp in fps.items():
    print(f"  {role:20s} {fp[:32]}...")

In [ ]:
# Three-state co-attestor: pass / flag / block
# Build a synthetic reference distribution (in production: from healthy runs)
ref = PerAgentToolFrequency(
    counts_by_role={
        "DataAgent": {"load_dataset": 1, "data_summary": 1,
                      "eda_correlation": 1, "preprocess_data": 1},
        "TrainingAgent": {"train_model": 5, "leaderboard": 1},
        "DiagnosticsAgent": {"diagnose_robustness": 2, "explain_importance": 2},
        "FairnessAgent": {"diagnose_fairness": 2},
    },
    total_calls=16,
)
attestor = MultiAgentCoAttestor(orchestrator, reference_freq=ref)

# Case 1: clean call -> pass
v1 = attestor.evaluate("DataAgent", "load_dataset",
                       {"name": "adult"},
                       recent_window=["load_dataset", "data_summary",
                                      "eda_correlation", "preprocess_data",
                                      "data_summary"])
print(f"clean call (DataAgent/load_dataset): verdict={v1.verdict} "
      f"psi={v1.psi_value:.4f}")

# Case 2: structural fail -> block
v2 = attestor.evaluate("DataAgent", "train_model", {"dataset": "adult"})
print(f"forbidden (DataAgent/train_model):    verdict={v2.verdict}")
print(f"  reasons: {v2.blocking_reasons}")

# Case 3: unknown alias -> block
v3 = attestor.evaluate("DataAgent", "train_model_v2", {})
print(f"unknown alias (DataAgent/train_model_v2): verdict={v3.verdict}")
print(f"  reasons: {v3.blocking_reasons}")

---
## Lab 17.4 --- Audit-Defensible Hardening

OOD validation, adversarial sweep, naive-baseline contrast,
exhaustive invariants proof, replay regression, audit log,
and HMAC-signed bundle.

In [ ]:
from knowlytix.harness.governance import (
    adult_income_multi_agent_cohort,
    large_adult_income_multi_agent_cohort,
    validate_multi_agent,
    build_multi_agent_adversarial_cohort,
    sweep_multi_agent_adversarial,
    SingleAgentBaseline, NoPermissionCheckBaseline, PromptOnlyBaseline,
    compare_to_orchestrator,
    verify_multi_agent_invariants,
    replay_multi_agent_battery,
    to_multi_agent_audit_record, replay_multi_agent_audit_log,
    build_multi_agent_bundle, MultiAgentOperatingPoint,
)

# 1. Invariants proof
inv = verify_multi_agent_invariants(orchestrator)
print(f"Invariants: {inv.n_pass}/{inv.n_cells} pass; {inv.n_violations} violations")

# 2. Bundle build
cohort = adult_income_multi_agent_cohort()
bundle = build_multi_agent_bundle(
    orch=orchestrator, cohort=cohort,
    source_dataset="adult_income",
    operating_point=MultiAgentOperatingPoint(
        min_adversarial_safety_rate=0.95,
        min_ood_decision_match_rate=0.85,
        min_alt_validator_agreement=1.0,
    ),
)
print(f"\nbundle_id={bundle.bundle_id}")
print(f"cohort_hash[:16]={bundle.cohort_hash[:16]}")
print(f"signing_method={bundle.signing_method}")

In [ ]:
# OOD validation (block + deny only for fast notebook execution)
big = large_adult_income_multi_agent_cohort(n_per_class=90, seed=42)
fast = [p for p in big if p.expected_decision in ("block", "deny")]
ood = validate_multi_agent(orchestrator, fast)
print(f"OOD on large block+deny cohort: matches={ood.decision_matches}/{ood.n}")

# Adversarial sweep
adv = build_multi_agent_adversarial_cohort(n_per_class=20, seed=42)
adv_report = sweep_multi_agent_adversarial(orchestrator, adv)
print(f"\nAdversarial: {adv_report.n_safe}/{adv_report.n_probes} = {adv_report.safety_rate:.4f}")
for cls, m in sorted(adv_report.per_class.items()):
    print(f"  {cls:42s}: {int(m['n_safe']):2d}/{int(m['n']):2d}")

# Replay regression
r1 = replay_multi_agent_battery(orchestrator, fast, bundle_id=bundle.bundle_id)
r2 = replay_multi_agent_battery(orchestrator, fast, bundle_id=bundle.bundle_id)
print(f"\nReplay battery: hash[:16]={r1.snapshots_hash[:16]}; byte-identical={r1.snapshots_hash == r2.snapshots_hash}")

In [ ]:
# Naive-baseline contrast
from knowlytix.harness.governance.agents.multi_agent_validation import _observe
decisions = {p.probe_id: _observe(orchestrator, p) for p in fast}

for cls in (SingleAgentBaseline, NoPermissionCheckBaseline, PromptOnlyBaseline):
    b = cls()
    contrast = compare_to_orchestrator(decisions, fast, b)
    print(f"vs {b.name:30s}: orch={contrast.n_correct_orchestrator}, "
          f"baseline={contrast.n_correct_baseline}, uplift=+{contrast.uplift}")

# Audit log replayable from log alone
rec_allow = to_multi_agent_audit_record(orchestrator, "DataAgent", "load_dataset",
                                        {"name": "adult"}, bundle=bundle)
rec_deny  = to_multi_agent_audit_record(orchestrator, "DataAgent", "train_model_v2",
                                        {}, bundle=bundle)
print(f"\nallow record: replayed={replay_multi_agent_audit_log(rec_allow.to_dict())}")
print(f"deny  record: replayed={replay_multi_agent_audit_log(rec_deny.to_dict())}")

---
## Lab 17.5 --- Production Hardening

Independent alt-validator + latency benchmarks.

In [ ]:
from knowlytix.harness.governance import (
    assert_orchestrator_alt_agreement,
    bench_can_execute, bench_handoff, bench_test_boundary,
)

alt = assert_orchestrator_alt_agreement(orchestrator, raise_on_disagreement=False)
print(f"alt-validator: {alt.n_agree}/{alt.n} = {alt.agreement_rate:.4f}")

b1 = bench_can_execute(orchestrator, n=1000)
b2 = bench_handoff(orchestrator, n=500)
b3 = bench_test_boundary(orchestrator, n=500)
for b in (b1, b2, b3):
    print(f"latency {b.operation:25s}: p50={b.p50_us:6.2f}us p95={b.p95_us:6.2f}us p99={b.p99_us:6.2f}us  SLO={int(b.slo_us)}us")